# PolySight Seg — verificación e inferencia

Este notebook clona una revisión fija, ejecuta los contratos del repositorio, verifica `best.pt` y segmenta una imagen. No entrena, no evalúa test y no modifica el modelo oficial.

In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import sys

REPOSITORY = 'https://github.com/christianbueno1/polysight-seg.git'
REPO_REF = 'c46d252b174546782291d9970b87190ce1ab0da1'
WORKSPACE = Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_ROOT = WORKSPACE / 'polysight-seg'
EXPECTED_CHECKPOINT_SHA256 = 'a3900c2db01e9e17fa7fedce12da94274d8995284f1c37f6e653df402919361b'
print({'python': sys.version, 'repo_ref': REPO_REF})

In [ ]:
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', REPO_REF], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run(['bash', 'scripts/validate_local.sh'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_inference_components.py', '-v'], check=True)

## Cargar y verificar el checkpoint

Por defecto se solicita `best.pt`. También se puede definir `CHECKPOINT_DRIVE_PATH` con una ruta de Google Drive. El archivo siempre se compara con el SHA-256 externo antes de crear el sidecar requerido por el cargador estricto.

In [ ]:
CHECKPOINT_DRIVE_PATH = ''  # Ejemplo: /content/drive/MyDrive/polysight/best.pt
CHECKPOINT_LOCAL_PATH = ''  # Para Jupyter/Kaggle; tiene prioridad sobre upload
checkpoint_path = PROJECT_ROOT / 'checkpoints/colab/best.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

if CHECKPOINT_LOCAL_PATH:
    shutil.copy2(CHECKPOINT_LOCAL_PATH, checkpoint_path)
elif CHECKPOINT_DRIVE_PATH:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2(CHECKPOINT_DRIVE_PATH, checkpoint_path)
else:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError('Define CHECKPOINT_LOCAL_PATH fuera de Colab') from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Sube exactamente el archivo best.pt')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'best.pt':
        raise RuntimeError('El archivo esperado se llama best.pt')
    checkpoint_path.write_bytes(content)

digest = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
if digest != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f'SHA-256 inesperado: {digest}')
checkpoint_path.with_suffix('.pt.sha256').write_text(f'{digest}  best.pt\n', encoding='utf-8')
print({'checkpoint': str(checkpoint_path), 'sha256': digest, 'size_mb': round(checkpoint_path.stat().st_size / 2**20, 1)})

In [ ]:
import torch
from polysight_seg.inference import load_verified_model

model, device, data_config, evaluation_config = load_verified_model(
    PROJECT_ROOT, checkpoint_path, device='auto'
)
print({
    'device': str(device),
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'selected_epoch': evaluation_config['checkpoint']['selected_epoch'],
    'threshold': evaluation_config['prediction']['threshold'],
})

## Inferencia de una imagen

Sube una imagen endoscópica JPEG o PNG. La salida es experimental y no debe utilizarse para decisiones clínicas.

In [ ]:
from io import BytesIO
import matplotlib.pyplot as plt
from PIL import Image
from polysight_seg.inference import predict_image

IMAGE_LOCAL_PATH = REPO_DIR / 'examples/segmentation/images/5514c1b0-6367-41ed-aafd-4444290904c4.jpg'
if IMAGE_LOCAL_PATH:
    image_name = Path(IMAGE_LOCAL_PATH).name
    image_bytes = Path(IMAGE_LOCAL_PATH).read_bytes()
else:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError('Define IMAGE_LOCAL_PATH fuera de Colab') from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Sube exactamente una imagen')
    image_name, image_bytes = next(iter(uploaded.items()))
image = Image.open(BytesIO(image_bytes)).convert('RGB')
result = predict_image(
    model, image, data_config, device,
    threshold=float(evaluation_config['prediction']['threshold']),
)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image); axes[0].set_title('Imagen')
axes[1].imshow(result.probability, cmap='viridis', vmin=0, vmax=1); axes[1].set_title('Probabilidad')
axes[2].imshow(result.mask, cmap='gray', vmin=0, vmax=255); axes[2].set_title('Máscara')
axes[3].imshow(result.overlay); axes[3].set_title('Overlay')
for axis in axes: axis.axis('off')
plt.tight_layout()
print({'image': image_name, 'original_size': result.original_size, 'foreground_fraction': result.foreground_fraction})